## Environments

In [ ]:
!pip install -q librosa easydict packaging \
                hear21passt timm torchcodec

In [ ]:
from abc import ABC, abstractmethod
from collections import Counter, OrderedDict
from datetime import datetime
from os.path import exists, join
from pathlib import Path
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Union

import copy
import gc
import json
import shutil
import logging
import math
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import random
import re
import seaborn as sns
import tarfile
import time
import warnings
import yaml
import zipfile

import librosa
import matplotlib.pyplot as plt
import torch
import torchaudio
import wandb

from dotenv import load_dotenv
from easydict import EasyDict

# from google.colab import userdata
from huggingface_hub import login as hf_login, snapshot_download
from IPython.display import Audio, display
from nltk.corpus import wordnet
from torch import Tensor
from torch.nn.functional import dropout, linear, pad, softmax
from torch.nn.init import constant_
from torch.nn.modules.linear import Linear
from torch.nn.modules.module import Module
from torch.utils.data import Dataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    ClapConfig,
    ClapFeatureExtractor,
    ClapModel,
    ClapProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    pipeline,
)

import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.overrides import has_torch_function, handle_torch_function
except:
    from torch._overrides import has_torch_function, handle_torch_function


load_dotenv()


HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
# HF_TOKEN = userdata.get('HF_TOKEN')
PROJECT_NAME = ""
WANDB_PROJECT_NAME = "[DCASE2026] Task6"
# WANDB_API_KEY = userdata.get('WANDB_API_KEY')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
wandb.login(key=WANDB_API_KEY)
hf_login(HF_TOKEN)

In [ ]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

In [ ]:
# LOCAL_DIR = Path("/content/drive/MyDrive/Dataset/DCASE2026").resolve()
# DATA_DIR = LOCAL_DIR / "CLOTHO-MOMENT"

LOCAL_DIR = Path("./").resolve()
DATA_DIR = LOCAL_DIR / "data"

os.listdir(DATA_DIR)

In [ ]:
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
FEATURES_DIR = DATA_DIR / "features"

# print(f"Examples from train dir: {os.listdir(TRAIN_DIR)[:5]}")
# print(f"Examples from validation dir: {os.listdir(VAL_DIR)[:5]}")
# print(f"Examples from test dir: {os.listdir(TEST_DIR)[:5]}")
# print(f"Examples from pre-processed dir: {os.listdir(PREPROCESSED_DIR)[:5]}")

## Experiments


In [ ]:
def get_run_name(prefix, lr: float = 2e-5, batch_size: int = 256):
    now = datetime.now().strftime("%m%d-%H%M")
    return f"{prefix}_lr{lr}_bs{batch_size}_{now}"


get_run_name(prefix="test-run-name")

In [ ]:
def clear_gpu_cache():
    print(f"[Before] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[Before] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

    # Clear GPU cache
    torch.cuda.empty_cache()
    # Run garbage collector
    gc.collect()

    # Verify memory is cleared
    print(f"[After] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[After] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# Verify memory is cleared
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
def copy_dir_with_progress(src, dst):
    """
    Copies a directory recursively and displays a tqdm progress bar.
    """
    src_path = Path(src)
    dst_path = Path(dst)

    if not src_path.exists():
        print(f"Error: Source directory '{src}' does not exist.")
        return

    # 1. Scan and count all files (ignoring directories for the count)
    print(f"Scanning '{src}' for files...")
    all_files = [f for f in src_path.rglob("*") if f.is_file()]
    total_files = len(all_files)

    if total_files == 0:
        print("No files found to copy.")
        return

    print(f"Found {total_files} files. Starting copy...")

    # 2. Copy files with tqdm progress bar
    for item in tqdm(all_files, desc="Copying Data", unit="file"):
        # Calculate the relative path to maintain the exact folder structure
        relative_path = item.relative_to(src_path)
        dest_item = dst_path / relative_path

        # Ensure the destination subdirectory exists before copying
        dest_item.parent.mkdir(parents=True, exist_ok=True)

        # Copy the file along with its metadata
        shutil.copy2(item, dest_item)

### Basic Utils

In [ ]:
class WandbLogger:
    def __init__(
        self, project_name: str, run_name: str, config: dict = None, entity: str = None
    ):
        """
        Initializes the W&B run.
        """
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=config,
            entity=entity,
            reinit=True,
        )
        self.best_accuracy = 0.0

    def log_metrics(self, metrics, step, prefix="eval"):
        """
        Logs a dictionary of metrics.

        Args:
            metrics (dict): The dictionary from your eval function.
            step (int): Current global step or epoch.
            prefix (str): Dashboard grouping (e.g., 'train' or 'eval').
        """
        # Format keys: {'loss': 0.5} -> {'eval/loss': 0.5}
        log_dict = {f"{prefix}/{k}": v for k, v in metrics.items()}

        # Log to W&B
        self.run.log(log_dict, step=step)

        # Optional: Track Best Metric Logic
        if "accuracy" in metrics:
            if metrics["accuracy"] > self.best_accuracy:
                self.best_accuracy = metrics["accuracy"]
                self.run.summary["best_accuracy"] = self.best_accuracy
                print(f"New best accuracy: {self.best_accuracy:.4f}")

    def log_artifact(self, model_path, name="model-checkpoint"):
        """Save your model file to W&B"""
        artifact = wandb.Artifact(name, type="model")
        artifact.add_file(model_path)
        self.run.log_artifact(artifact)

    def finish(self):
        """Closes the W&B run"""
        self.run.finish()

In [ ]:
def write_log(opt, epoch_i, loss_meters, metrics=None, mode="train"):
    # log
    if mode == "train":
        to_write = opt.train_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i + 1,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
        )
        filename = opt.train_log_filepath
    else:
        to_write = opt.eval_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
            eval_metrics_str=json.dumps(metrics),
        )
        filename = opt.eval_log_filepath

    with open(filename, "a") as f:
        f.write(to_write)


def save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt):
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch_i,
        "opt": opt,
    }
    torch.save(checkpoint, opt.ckpt_filepath)


def rename_latest_to_best(latest_file_paths):
    best_file_paths = [e.replace("latest", "best") for e in latest_file_paths]
    for src, tgt in zip(latest_file_paths, best_file_paths):
        os.renames(src, tgt)


def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def save_json(data, filename, save_pretty=False, sort_keys=False):
    with open(filename, "w") as f:
        if save_pretty:
            f.write(json.dumps(data, indent=4, sort_keys=sort_keys))
        else:
            json.dump(data, f)


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(l.strip("\n")) for l in f.readlines()]


def save_jsonl(data, filename):
    """data is a list"""
    with open(filename, "w") as f:
        f.write("\n".join([json.dumps(e) for e in data]))


def save_lines(list_of_str, filepath):
    with open(filepath, "w") as f:
        f.write("\n".join(list_of_str))


def read_lines(filepath):
    with open(filepath, "r") as f:
        return [e.strip("\n") for e in f.readlines()]


def read_yaml(file_path: Union[str, Path]) -> Dict[str, Any]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"YAML file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = yaml.safe_load(f)
            return data if data is not None else {}
        except yaml.YAMLError as e:
            raise yaml.YAMLError(f"Error parsing YAML file {file_path}: {e}")


def mkdirp(p):
    if not os.path.exists(p):
        os.makedirs(p)


def flat_list_of_lists(l):
    """flatten a list of lists [[1,2], [3,4]] to [1,2,3,4]"""
    return [item for sublist in l for item in sublist]


def convert_to_seconds(hms_time):
    """convert '00:01:12' to 72 seconds.
    :hms_time (str): time in comma separated string, e.g. '00:01:12'
    :return (int): time in seconds, e.g. 72
    """
    times = [float(t) for t in hms_time.split(":")]
    return times[0] * 3600 + times[1] * 60 + times[2]


def get_video_name_from_url(url):
    return url.split("/")[-1][:-4]


def merge_dicts(list_dicts):
    merged_dict = list_dicts[0].copy()
    for i in range(1, len(list_dicts)):
        merged_dict.update(list_dicts[i])
    return merged_dict


def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def make_zipfile(
    src_dir,
    save_path,
    enclosing_dir="",
    exclude_dirs=None,
    exclude_extensions=None,
    exclude_dirs_substring=None,
):
    """make a zip file of root_dir, save it to save_path.
    exclude_paths will be excluded if it is a subdir of root_dir.
    An enclosing_dir is added is specified.
    """
    abs_src = os.path.abspath(src_dir)
    with zipfile.ZipFile(save_path, "w") as zf:
        for dirname, subdirs, files in os.walk(src_dir):
            if exclude_dirs is not None:
                for e_p in exclude_dirs:
                    if e_p in subdirs:
                        subdirs.remove(e_p)
            if exclude_dirs_substring is not None:
                to_rm = []
                for d in subdirs:
                    if exclude_dirs_substring in d:
                        to_rm.append(d)
                for e in to_rm:
                    subdirs.remove(e)
            arcname = os.path.join(enclosing_dir, dirname[len(abs_src) + 1 :])
            zf.write(dirname, arcname)
            for filename in files:
                if exclude_extensions is not None:
                    if os.path.splitext(filename)[1] in exclude_extensions:
                        continue  # do not zip it
                absname = os.path.join(dirname, filename)
                arcname = os.path.join(enclosing_dir, absname[len(abs_src) + 1 :])
                zf.write(absname, arcname)


class AverageMeter(object):
    """Computes and stores the average and current/max/min value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10

    def update(self, val, n=1):
        self.max = max(val, self.max)
        self.min = min(val, self.min)
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def dissect_by_lengths(np_array, lengths, dim=0, assert_equal=True):
    """Dissect an array (N, D) into a list a sub-array,
    np_array.shape[0] == sum(lengths), Output is a list of nd arrays, singlton dimention is kept"""
    if assert_equal:
        assert len(np_array) == sum(lengths)
    length_indices = [
        0,
    ]
    for i in range(len(lengths)):
        length_indices.append(length_indices[i] + lengths[i])
    if dim == 0:
        array_list = [
            np_array[length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 1:
        array_list = [
            np_array[:, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 2:
        array_list = [
            np_array[:, :, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    else:
        raise NotImplementedError
    return array_list


def get_ratio_from_counter(counter_obj, threshold=200):
    keys = counter_obj.keys()
    values = counter_obj.values()
    filtered_values = [counter_obj[k] for k in keys if k > threshold]
    return float(sum(filtered_values)) / sum(values)


def get_counter_dist(counter_object, sort_type="none"):
    _sum = sum(counter_object.values())
    dist = {k: float(f"{100 * v / _sum:.2f}") for k, v in counter_object.items()}
    if sort_type == "value":
        dist = OrderedDict(sorted(dist.items(), reverse=True))
    return dist


def get_show_name(vid_name):
    """
    get tvshow name from vid_name
    :param vid_name: video clip name
    :return: tvshow name
    """
    show_list = ["friends", "met", "castle", "house", "grey"]
    vid_name_prefix = vid_name.split("_")[0]
    show_name = vid_name_prefix if vid_name_prefix in show_list else "bbt"
    return show_name


def get_abspaths_by_ext(dir_path, ext=(".jpg",)):
    """Get absolute paths to files in dir_path with extensions specified by ext.
    Note this function does work recursively.
    """
    if isinstance(ext, list):
        ext = tuple(ext)
    if isinstance(ext, str):
        ext = tuple(
            [
                ext,
            ]
        )
    filepaths = [
        os.path.join(root, name)
        for root, dirs, files in os.walk(dir_path)
        for name in files
        if name.endswith(tuple(ext))
    ]
    return filepaths


def get_basename_no_ext(path):
    """'/data/movienet/240p_keyframe_feats/tt7672188.npz' --> 'tt7672188'"""
    return os.path.splitext(os.path.split(path)[1])[0]


def dict_to_markdown(d, max_str_len=120):
    # convert list into its str representation
    d = {k: v.__repr__() if isinstance(v, list) else v for k, v in d.items()}
    # truncate string that is longer than max_str_len
    if max_str_len is not None:
        d = {k: v[-max_str_len:] if isinstance(v, str) else v for k, v in d.items()}
    return pd.DataFrame(d, index=[0]).transpose().to_markdown()

In [ ]:
preprocessd_sample = load_jsonl(
    str(PREPROCESSED_DIR / "clotho_moment_train_release.jsonl")
)

preprocessd_sample[0]

In [ ]:
# y_sample, sr_sample = librosa.load(
#     str(TRAIN_DIR / "Venice_40_640.wav"),
#     sr=None
# )

y_sample, sr_sample = torchaudio.load(str(TRAIN_DIR / "Venice_40_640.wav"))

In [ ]:
y_sample = y_sample.numpy()

In [ ]:
sr_sample

In [ ]:
y_sample

In [ ]:
display(Audio(data=y_sample, rate=sr_sample))

In [ ]:
plt.figure()
librosa.display.waveshow(y_sample, sr=sr_sample, alpha=0.7)
plt.title(f"Waveform – qid {preprocessd_sample[0]['qid']}", fontsize=14, weight="bold")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

In [ ]:
# n_fft = 1024
# hop_length = 256

# mel_spec = librosa.feature.melspectrogram(
#     y=y_sample,
#     sr=sr_sample,
#     n_fft=n_fft,
#     hop_length=hop_length,
#     n_mels=40,
#     fmax=sr_sample / 2,
# )
# log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
# plt.figure()
# librosa.display.specshow(
#     log_mel_spec,
#     sr=sr_sample,
#     hop_length=hop_length,
#     x_axis="time",
#     y_axis="mel",
#     cmap="viridis",
# )
# plt.title("Log‑Mel Spectrogram", fontsize=14, weight="bold")
# plt.colorbar(format="%+2.0f dB")
# plt.tight_layout()
# plt.show()

### Span Utils

In [ ]:
def span_xx_to_cxw(xx_spans):
    """
    Args:
        xx_spans: tensor, (#windows, 2) or (..., 2), each row is a window of format (st, ed)

    Returns:
        cxw_spans: tensor, (#windows, 2), each row is a window of format (center=(st+ed)/2, width=(ed-st))
    >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
    >>> span_xx_to_cxw(spans)
    tensor([[0.5000, 1.0000],
        [0.3000, 0.2000]])
    >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
    >>> span_xx_to_cxw(spans)
    tensor([[[0.5000, 1.0000],
         [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)


def span_cxw_to_xx(cxw_spans):
    """
    Args:
        cxw_spans: tensor, (#windows, 2) or (..., 2), the last dim is a row denoting a window of format (center, width)

    >>> spans = torch.Tensor([[0.5000, 1.0000], [0.3000, 0.2000]])
    >>> span_cxw_to_xx(spans)
    tensor([[0.0000, 1.0000],
        [0.2000, 0.4000]])
    >>> spans = torch.Tensor([[[0.5000, 1.0000], [0.3000, 0.2000]]])
    >>> span_cxw_to_xx(spans)
    tensor([[[0.0000, 1.0000],
        [0.2000, 0.4000]]])
    """
    x1 = cxw_spans[..., 0] - 0.5 * cxw_spans[..., 1]
    x2 = cxw_spans[..., 0] + 0.5 * cxw_spans[..., 1]
    return torch.stack([x1, x2], dim=-1)


def temporal_iou(spans1, spans2):
    """
    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        iou: (N, M) torch.Tensor
        union: (N, M) torch.Tensor
    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> temporal_iou(test_spans1, test_spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = torch.max(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.min(spans1[:, None, 1], spans2[:, 1])  # (N, M)

    inter = (right - left).clamp(min=0)  # (N, M)
    union = areas1[:, None] + areas2 - inter  # (N, M)

    iou = inter / union
    return iou, union


def temporal_intersection_over_pred(gt_spans, pred_spans):
    """intersection over the second input spans
    Args:
        gt_spans: (N, 2),
        pred_spans: (M, 2)

    Returns:

    """
    left = torch.max(gt_spans[:, None, 0], pred_spans[:, 0])
    right = torch.min(gt_spans[:, None, 1], pred_spans[:, 1])

    inter = (right - left).clamp(min=0)  # (N, M)
    inter_over_pred = inter / (pred_spans[:, 1] - pred_spans[:, 0])
    return inter_over_pred


def generalized_temporal_iou(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    assert (spans1[:, 1] >= spans1[:, 0]).all()
    assert (spans2[:, 1] >= spans2[:, 0]).all()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area


def generalized_temporal_iou_(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area

### Tensor Utils

In [ ]:
def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def pad_sequences_2d(sequences, dtype=torch.long):
    """Pad a double-nested list or a sequence of n-d torch tensor into a (n+1)-d tensor,
        only allow the first two dims has variable lengths
    Args:
        sequences: list(n-d tensor or list)
        dtype: torch.long for word indices / torch.float (float32) for other cases
    Returns:
    Examples:
        >>> test_data_list = [[[1, 3, 5], [3, 7, 4, 1]], [[98, 34, 11, 89, 90], [22], [34, 56]],]
        >>> pad_sequences_2d(test_data_list, dtype=torch.long)  # torch.Size([2, 3, 5])
        >>> test_data_3d = [torch.randn(2,2,4), torch.randn(4,3,4), torch.randn(1,5,4)]
        >>> pad_sequences_2d(test_data_3d, dtype=torch.float)  # torch.Size([2, 3, 5])
        >>> test_data_3d2 = [[torch.randn(2,4), ], [torch.randn(3,4), torch.randn(5,4)]]
        >>> pad_sequences_2d(test_data_3d2, dtype=torch.float)  # torch.Size([2, 3, 5])
    # TODO add support for numpy array
    """
    bsz = len(sequences)
    para_lengths = [len(seq) for seq in sequences]
    max_para_len = max(para_lengths)
    sen_lengths = [[len(word_seq) for word_seq in seq] for seq in sequences]
    max_sen_len = max([max(e) for e in sen_lengths])

    if isinstance(sequences[0], torch.Tensor):
        extra_dims = sequences[0].shape[2:]
    elif isinstance(sequences[0][0], torch.Tensor):
        extra_dims = sequences[0][0].shape[1:]
    else:
        sequences = [
            [torch.Tensor(word_seq, dtype=dtype) for word_seq in seq]
            for seq in sequences
        ]
        extra_dims = ()

    padded_seqs = torch.zeros(
        (bsz, max_para_len, max_sen_len) + extra_dims, dtype=dtype
    )
    mask = torch.zeros(bsz, max_para_len, max_sen_len).float()

    for b_i in range(bsz):
        for sen_i, sen_l in enumerate(sen_lengths[b_i]):
            padded_seqs[b_i, sen_i, :sen_l] = sequences[b_i][sen_i]
            mask[b_i, sen_i, :sen_l] = 1
    return padded_seqs, mask  # , sen_lengths

In [ ]:
# Example

test_data_list = [[1, 2, 3], [1, 2], [3, 4, 7, 9]]
pad_sequences_1d(test_data_list, dtype=torch.long)

In [ ]:
# Ref: https://github.com/nttcslab/m2d/tree/master/examples

# !pip install -q einops nnAudio
# !wget https://raw.githubusercontent.com/nttcslab/m2d/master/examples/portable_m2d.py
# !wget https://github.com/nttcslab/m2d/releases/download/v0.5.0/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip

# with zipfile.ZipFile("m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip", "r") as zip_ref:
#     zip_ref.extractall(".")

# !find m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025 -name *.pth

In [ ]:
# from portable_m2d import PortableM2D
# model = PortableM2D(weight_file='m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth')
# model.eval()

# # model.to(DEVICE)

# audio = torch.from_numpy(y_sample).unsqueeze(0)
# embedding = model(audio)

# embedding # [1, 513, 3840] => [:, : 768 * 5]

In [ ]:
# clap_model_id = "laion/clap-htsat-unfused"
clap_model_id = "laion/clap-htsat-fused"
clap_model = AutoModel.from_pretrained(clap_model_id).to(DEVICE)
# clap_tokenizer = AutoTokenizer.from_pretrained(clap_model_id)
clap_processor = AutoProcessor.from_pretrained(clap_model_id)

clap_processor.feature_extractor = ClapFeatureExtractor(sampling_rate=sr_sample)

### Data Preparing

Include 3 steps:

1. DataProcessor: Imply models for features extraction, augmentations, ...
2. Dataset: Use DataProcessor, defining input and output of the model
3. DataLoader: Use Dataset, load data to GPU, ..


#### Data Augmentation

In [ ]:
class BaseAugmentator:
    def __call__(self, payload: any, *args, **kwargs) -> any:
        raise NotImplementedError

In [ ]:
class TextAugmentor(BaseAugmentator):
    def __call__(self, payload: str) -> str:
        # Apply one random text augmentation
        aug = random.choice(
            [self.synonym_replacement, self.random_insertion, self.add_template]
        )
        return aug(payload)

    def get_synonyms(self, word):
        """Get synonyms for a word using WordNet."""
        synonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                synonyms.add(lemma.name())
        if word in synonyms:
            synonyms.remove(word)
        return list(synonyms)

    def synonym_replacement(self, text, n=1):
        """Replace n words in the text with their synonyms."""
        words = text.split()
        new_words = words.copy()
        random_word_list = list(set([word for word in words if word.isalnum()]))
        random.shuffle(random_word_list)
        num_replaced = 0
        for random_word in random_word_list:
            synonyms = self.get_synonyms(random_word)
            if len(synonyms) >= 1:
                synonym = random.choice(synonyms)
                synonym = synonym.replace("_", " ")
                new_words = [
                    synonym if word == random_word else word for word in new_words
                ]
                num_replaced += 1
            if num_replaced >= n:
                break
        return " ".join(new_words)

    # def random_deletion(text, p=0.1):
    #     """Randomly delete words with probability p."""
    #     words = text.split()
    #     if len(words) == 1:
    #         return text
    #     new_words = []
    #     for word in words:
    #         r = random.uniform(0, 1)
    #         if r > p:
    #             new_words.append(word)
    #     if len(new_words) == 0:
    #         rand_int = random.randint(0, len(words)-1)
    #         return words[rand_int]
    #     return ' '.join(new_words)

    # def random_swap(text, n=1):
    #     """Randomly swap two words in the text n times."""
    #     words = text.split()
    #     new_words = words.copy()
    #     for _ in range(n):
    #         if len(new_words) < 2:
    #             break
    #         idx1, idx2 = random.sample(range(len(new_words)), 2)
    #         new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    #     return ' '.join(new_words)

    def random_insertion(self, text, n=3):
        """Randomly insert n words into the text."""
        words = text.split()
        new_words = words.copy()
        for _ in range(n):
            self.add_word(new_words)
        return " ".join(new_words)

    def add_word(self, new_words):
        """Helper function to add a random word."""
        synonyms = []
        counter = 0
        while len(synonyms) < 1:
            random_word = new_words[random.randint(0, len(new_words) - 1)]
            synonyms = self.get_synonyms(random_word)
            counter += 1
            if counter >= 10:
                return
        random_synonym = random.choice(synonyms)
        random_idx = random.randint(0, len(new_words) - 1)
        random_synonym = random_synonym.replace("_", " ")
        new_words.insert(random_idx, random_synonym)

    def add_template(self, text: str) -> str:
        templates = [
            # Original templates
            "The sound of {caption_lower}",
            "Audio recording of {caption_lower}",
            "A clip where {caption_lower}",
            "You can hear {caption_lower}",
            "An audio where {caption_lower}",
            # New additions: Action & Context
            "A scene featuring {caption_lower}",
            "The distinct noise of {caption_lower}",
            "In this recording, {caption_lower}",
            "A high-quality capture of {caption_lower}",
            "Listen closely to {caption_lower}",
            # New additions: Short & Descriptive
            "Ambience of {caption_lower}",
            "The acoustic profile of {caption_lower}",
            "Background audio of {caption_lower}",
            "A demonstration of {caption_lower}",
            "The sonorous quality of {caption_lower}",
            # New additions: Narrative
            "This audio track contains {caption_lower}",
            "The listener can perceive {caption_lower}",
            "A snippet showcasing {caption_lower}",
            "Experience the sound of {caption_lower}",
            "The atmosphere is filled with {caption_lower}",
        ]

        template = random.choice(templates)
        return template.format(caption=text, caption_lower=text.lower())

In [ ]:
class AudioAugmentor(BaseAugmentator):
    """Safe audio augmentations for audio-text retrieval."""

    def __init__(self, sr: int = 44100):
        self.sr = sr
        self.augmentations = [
            self.add_noise,
            self.random_gain,
            self.time_shift,
            self.polarity_inversion,
            self.speed_perturb,
            self.spec_augment_on_waveform,
        ]

    def __call__(self, payload: torch.Tensor) -> torch.Tensor:
        # Apply 1-3 random augmentations
        n_augs = random.randint(1, 3)
        chosen: List[Callable] = random.sample(
            self.augmentations, min(n_augs, len(self.augmentations))
        )
        for aug in chosen:
            waveform = aug(payload)
        return waveform

    def add_noise(
        self, waveform: torch.Tensor, snr_db_range: tuple[float, float] = (15, 40)
    ) -> torch.Tensor:
        snr_db = random.uniform(*snr_db_range)
        noise = torch.randn_like(waveform)
        signal_power = waveform.norm(p=2)
        noise_power = noise.norm(p=2)
        if noise_power == 0:
            return waveform
        scale = signal_power / (10 ** (snr_db / 20) * noise_power)
        return waveform + scale * noise

    def random_gain(
        self, waveform: torch.Tensor, min_db: float = -6, max_db: float = 6
    ) -> torch.Tensor:
        gain_db = random.uniform(min_db, max_db)
        return waveform * (10 ** (gain_db / 20))

    def time_shift(
        self, waveform: torch.Tensor, max_shift: float = 0.3
    ) -> torch.Tensor:
        shift = int(waveform.shape[-1] * random.uniform(-max_shift, max_shift))
        return torch.roll(waveform, shifts=shift, dims=-1)

    def polarity_inversion(self, waveform: torch.Tensor) -> torch.Tensor:
        return -waveform if random.random() > 0.5 else waveform

    def speed_perturb(
        self, waveform: torch.Tensor, factor_range: tuple[float, float] = (0.95, 1.05)
    ) -> torch.Tensor:
        factor = random.uniform(*factor_range)
        resampled = torchaudio.functional.resample(
            waveform, orig_freq=self.sr, new_freq=int(self.sr * factor)
        )
        return resampled

    def spec_augment_on_waveform(self, waveform: torch.Tensor) -> torch.Tensor:
        """Apply small random zero-out segments (simplified SpecAugment on waveform)."""
        clip_fraction = random.uniform(0.0, 0.05)
        clip_len = int(waveform.shape[-1] * clip_fraction)
        if clip_len > 0:
            start = random.randint(0, waveform.shape[-1] - clip_len)
            waveform = waveform.clone()
            waveform[..., start : start + clip_len] = 0
        return waveform

#### DataProcessor

1. Cleaning data
2. Embedding data
3. Save features (if possible)

In [ ]:
query_sample = preprocessd_sample[0].get("query")
query_sample

In [ ]:
text_augmenter = TextAugmentor()
# audio_augmenter = AudioAugmentor()

In [ ]:
text_augmenter(query_sample)

In [ ]:
# y_augment = audio_augmenter(torch.Tensor(y_sample)).numpy()

In [ ]:
# display(Audio(data=y_augment,
#               rate=sr_sample))


# plt.figure()
# librosa.display.waveshow(y_augment, sr=sr_sample, alpha=0.7)
# plt.title(f"Waveform – qid {preprocessd_sample[0]["qid"]}", fontsize=14, weight="bold")
# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.tight_layout()
# plt.show()

In [ ]:
y_sample.shape

In [ ]:
1323000 / 60

In [ ]:
window_length = sr_sample * 1  # 1 second
hop_length = sr_sample * 1  # Move by 1 second

# 2. Frame the raw audio WITHOUT padding
# This will strictly cut it into 60 consecutive 1-second pieces
chunks = librosa.util.frame(y_sample, frame_length=window_length, hop_length=hop_length)

# 3. Transpose to shape (60, samples_per_second)
chunks = chunks.T

In [ ]:
input_sample = clap_processor(
    audio=chunks,
    text=query_sample,
    padding="max_length",  # Forces it to stretch to max_length
    sampling_rate=sr_sample,
    return_tensors="pt",
    max_length=32,
)

input_sample = input_sample.to(DEVICE)

In [ ]:
clear_gpu_cache()

In [ ]:
text_features = clap_model.get_text_features(**input_sample)
audio_features = clap_model.get_audio_features(**input_sample)

In [ ]:
# Pass inputs to model
outputs = clap_model(**input_sample)

# Grab the final 1D summary vector for the text
text_embeddings = outputs.text_embeds

In [ ]:
clear_gpu_cache()

DEFAULT_SAMPLING_RATE = 32_000

clap_model_id = "laion/clap-htsat-fused"
clap_model = AutoModel.from_pretrained(clap_model_id).to(DEVICE)
clap_processor = AutoProcessor.from_pretrained(clap_model_id)

# Force the feature extractor to use your target sampling rate
clap_processor.feature_extractor = ClapFeatureExtractor(
    sampling_rate=DEFAULT_SAMPLING_RATE
)

In [ ]:
text_config = {
    "augmentor": TextAugmentor(),  # Assuming this is defined elsewhere
    "max_length": 32,
    "padding": "max_length",
}

audio_config = {
    "augmentor": AudioAugmentor(),  # Assuming this is defined elsewhere
    "sr": DEFAULT_SAMPLING_RATE,
    "window_length": DEFAULT_SAMPLING_RATE * 1,
    "hop_length": DEFAULT_SAMPLING_RATE,
}


"""
1.  **If you are matching general audio to text (e.g., "A jazz band playing"):** Stick with **Mean**. It captures the continuous nature of the music.
2.  **If you are looking for specific sound effects (e.g., "Glass breaking"):** Try **Max**. It will highlight the sharp transient of the shatter.
3.  **If you want to test it yourself:** You can actually combine them! A very common trick in Kaggle competitions is to do both and stick them together:
    ```python
    mean_pool = embeds.mean(dim=(2, 3))
    max_pool = embeds.amax(dim=(2, 3))

    # Shape becomes (60, 1536). You get the best of both worlds!
    combined = torch.cat([mean_pool, max_pool], dim=1)
    ```
"""
clear_gpu_cache()


class DataFeatureExtractor(object):
    def __init__(
        self,
        processor: Union[AutoProcessor, Any],
        model: Any,
        *,
        device: str = DEVICE,
        text_config: Dict = text_config,
        audio_config: Dict = audio_config,
    ):
        self.processor = processor
        self.model = model
        self.text_config = text_config
        self.audio_config = audio_config
        self.device = device

        # Ensure model is in eval mode so we don't calculate gradients
        self.model.eval()

    def __call__(
        self,
        payload: dict,
        *,
        sliding_window: int = 1,
        augment_strategy: Literal["none", "text", "audio", "both"] = "none",
    ) -> Dict:
        """
        Expected payload:
        {
          "query": "Man in gray top walks from outside to inside.",
          "audio": "RoripwjYFp8_360.0_510.0.wav"
        }
        """
        features = {"audio": {}}

        try:
            # TODO: add augmentation
            query = payload.get("query")
            audio_path = payload.get("audio")

            # 1. Load and prepare audio
            waveform, sr = self.read_audio(audio_path)

            # Local Context (1-second chunks)
            chunks = self.chunk_audio(waveform, window_length=sr * 1, hop_length=sr * 1)

            # 2. Process Inputs
            text_inputs = self.processor(
                text=query,
                padding=self.text_config.get("padding", "max_length"),
                truncation=True,
                max_length=self.text_config.get("max_length", 32),
                return_tensors="pt",
            ).to(self.device)

            global_audio_inputs = self.processor(
                audio=waveform,  # Hugging Face standard is 'audios'
                sampling_rate=sr,
                return_tensors="pt",
            ).to(self.device)

            local_audio_inputs = self.processor(
                audio=list(chunks),  # Processor expects a list of 1D arrays for batches
                sampling_rate=sr,
                return_tensors="pt",
            ).to(self.device)

            # 3. Extract Features using CLAP Model
            with torch.no_grad():
                text_outputs = self.model.get_text_features(**text_inputs)
                global_audio_outputs = self.model.get_audio_features(
                    **global_audio_inputs
                )
                local_audio_outputs = self.model.get_audio_features(
                    **local_audio_inputs
                )

            # 4. Extract the raw tensors
            def get_tensor(output):
                if isinstance(output, torch.Tensor):
                    return output
                # Using last_hidden_state as requested
                return output.last_hidden_state

            text_embeds = get_tensor(text_outputs)
            global_audio_embeds = get_tensor(global_audio_outputs)
            local_audio_embeds = get_tensor(local_audio_outputs)

            # 5. Dimensionality Reduction (Pooling)
            # Text: typically (batch, seq_len, hidden_dim). Pool across sequence (dim=1)
            if text_embeds.ndim == 3:
                text_embeds = text_embeds.mean(dim=1)

            # Audio: typically (batch, hidden_dim, freq, time) e.g., (60, 768, 2, 32). Pool across freq and time (dim=(2, 3))
            if global_audio_embeds.ndim == 4:
                global_audio_embeds = global_audio_embeds.mean(dim=(2, 3))
            elif (
                global_audio_embeds.ndim == 3
            ):  # Fallback if shape is (batch, seq, dim)
                global_audio_embeds = global_audio_embeds.mean(dim=1)

            if local_audio_embeds.ndim == 4:
                local_audio_embeds = local_audio_embeds.mean(dim=(2, 3))
            elif local_audio_embeds.ndim == 3:
                local_audio_embeds = local_audio_embeds.mean(dim=1)

            # 6. Store and move back to CPU as NumPy arrays
            features["text"] = text_embeds.cpu().numpy()
            features["audio"]["global"] = global_audio_embeds.cpu().numpy()
            features["audio"]["local"] = local_audio_embeds.cpu().numpy()

            return features

        except Exception as e:
            print(f"Error extracting features: {e}")
            return dict()

    def read_audio(self, path: str) -> tuple[np.ndarray, int]:
        waveform, sr = torchaudio.load(path)

        # Convert to Mono if stereo (take the mean across the channel dimension)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Squeeze the [1, samples] tensor to a 1D array [samples] for librosa
        waveform_np = waveform.squeeze().numpy()
        return waveform_np, sr

    def chunk_audio(
        self, waveform: np.ndarray, window_length: int, hop_length: int
    ) -> np.ndarray:

        # librosa.util.frame expects a 1D array.
        chunks = librosa.util.frame(
            waveform, frame_length=window_length, hop_length=hop_length
        )

        # Transpose to shape (num_chunks, samples_per_chunk)
        chunks = chunks.T
        return chunks

In [ ]:
processor = DataFeatureExtractor(processor=clap_processor, model=clap_model)
processor

In [ ]:
sample = preprocessd_sample[0]
payload = {
    "query": sample["query"],
    "audio": str(TRAIN_DIR / f"{sample['vid'].replace('.', '')}.wav"),
}


payload

In [ ]:
process_sample = processor(payload=payload)
process_sample

In [ ]:
process_sample["text"].shape

In [ ]:
process_sample["audio"]["local"].shape

In [ ]:
process_sample["audio"]["global"].shape

In [ ]:
def get_dir_size(path="."):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    # entry.stat() is cached on some systems, making this very fast
                    total_size += entry.stat().st_size
                elif entry.is_dir():
                    # Recursively call the function for subdirectories
                    total_size += get_dir_size(entry.path)
    except PermissionError:
        # Handle folders you don't have access to
        return 0
    return total_size


size_in_bytes = get_dir_size(str(TRAIN_DIR))
print(f"Total Size: {size_in_bytes} bytes")

In [ ]:
def format_size(bytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if bytes < 1024:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024


print(f"Total Size: {format_size(size_in_bytes)}")

In [ ]:
val_size_in_bytes = get_dir_size(str(VAL_DIR))
print(f"Total Size: {val_size_in_bytes} bytes")
print(f"Total Size: {format_size(val_size_in_bytes)}")


test_size_in_bytes = get_dir_size(str(TEST_DIR))
print(f"Total Size: {test_size_in_bytes} bytes")
print(f"Total Size: {format_size(test_size_in_bytes)}")

#### (Custom) Dataset

1. Prepare Dataset
2. Prepare Dataloader

In [ ]:
# class DCASET6Dataset(Dataset):
#     """One line in data loaded from data_path."
#     {
#       "qid": 7803,
#       "query": "Man in gray top walks from outside to inside.",
#       "duration": 150,
#       "vid": "RoripwjYFp8_360.0_510.0",
#       "relevant_clip_ids": [13, 14, 15, 16, 17],
#       "relevant_windows": [[26, 36]]
#     }
#     """
#   def __init__(self,
#                data_path: str | Path,
#                data_processor: DataFeatureExtractor,
#               q_feat_type: str = "last_hidden_state",
#               a_feat_type: str = "pann",
#               max_q_l: int = 32,
#               max_a_l: int = 75,
#               ctx_mode: str = "video",
#               clip_len: int = 2,
#               max_windows: int = 5,
#               span_loss_type: str = "l1",
#               load_labels: bool = True,
#                *,
#                mode: Literal['train', 'val', 'test'] = 'train',
#                text_config: Dict = text_config,
#                audio_config: Dict = audio_config,
#                )

#     if not isintance(self.data_path, Path):
#       data_path = Path(self.data_path)
#     self.data_path = data_path

#     self.data_processor = data_processor
#     self.q_feat_type = q_feat_type
#     self.a_feat_type = a_feat_type
#     self.max_q_l = max_q_l
#     self.max_a_l = max_a_l
#     self.ctx_mode = ctx_mode
#     self.clip_len = clip_len
#     self.max_windows = max_windows
#     self.span_loss_type = span_loss_type
#     self.load_labels = load_labels

#     self.data = self.load_data()
#     self.mode = mode.lower()
#     self.text_config = text_config
#     self.audio_config = audio_config


#   def load_data(self) -> List[Dict[str, Any]]:
#       datalist = load_jsonl(self.data_path)
#       return datalist

#   def __len__(self) -> int:
#       return len(self.data)

#   def __getitem__(self, index: int):
#       meta = self.data[index]

#       query = meta.get('query')
#       _audio_filename = meta.get('vid')
#       _root_dir = self.data_path.parent
#       audio_path = _root_dir / "train" / f"{_audio_filename.replace(".", "")}.wav"
#       model_inputs = dict()

#       data_features = self.data_processor(
#           payload={
#               "audio": audio_path,
#               "query": query
#           },
#       ) # shape: text:[1, 768]; audio_local:[sliding_window, 768], audio_global[1, 768]

#       model_inputs["query_feat"] = data_features.get('text') # (Dq, ) or (Lq, Dq)
#       model_inputs["audio_feat"] = data_features.get('audio').get('local')
#       model_inputs["audio_feat_global"] = data_features.get('audio').get('global')

#       tef_st = torch.arange(0, ctx_l, 1.0) / ctx_l
#       tef_ed = tef_st + 1.0 / ctx_l
#       tef = torch.stack([tef_st, tef_ed], dim=1)  # (Lv, 2)
#       model_inputs["audio_feat"] = torch.cat(
#           [model_inputs["audio_feat"], tef], dim=1
#       )

#     if self.load_labels:
#         model_inputs["span_labels"] = self.get_span_labels(
#             meta["relevant_windows"], ctx_l
#         )
#         (
#             model_inputs["saliency_pos_labels"],
#             model_inputs["saliency_neg_labels"],
#             model_inputs["saliency_all_labels"],
#         ) = self.get_saliency_labels_sub_as_query(
#             meta["relevant_windows"][0], ctx_l
#         )

#     return dict(meta=meta, model_inputs=model_inputs)

#   def get_span_labels(self, windows: List[List[float]], ctx_l: int) -> torch.Tensor:
#         """
#         windows: list([st, ed]) in seconds. E.g. [[26, 36]], corresponding st_ed clip_indices [[13, 17]] (inclusive)
#             Note a maximum of `self.max_windows` windows are used.
#         returns Tensor of shape (#windows, 2), each row is [center, width] normalized by video length
#         """
#         if len(windows) > self.max_windows:
#             random.shuffle(windows)
#             windows = windows[: self.max_windows]
#         if self.span_loss_type == "l1":
#             windows = torch.Tensor(windows) / (
#                 ctx_l * self.clip_len
#             )  # normalized windows in xx
#             windows = span_xx_to_cxw(windows)  # normalized windows in cxw
#         elif self.span_loss_type == "ce":
#             windows = torch.Tensor(
#                 [
#                     [
#                         int(w[0] / self.clip_len),
#                         min(int(w[1] / self.clip_len), ctx_l) - 1,
#                     ]
#                     for w in windows
#                 ]
#             ).long()  # inclusive
#         else:
#             raise NotImplementedError
#         return windows

#   def get_saliency_labels_sub_as_query(self, gt_window: List[float], ctx_l: int, max_n: int = 2) -> Tuple[List[int], List[int], np.ndarray]:

#         gt_st = int(gt_window[0] / self.clip_len)
#         gt_ed = max(0, min(int(gt_window[1] / self.clip_len), ctx_l) - 1)

#         if gt_st > gt_ed:
#             gt_st = gt_ed

#         if gt_st != gt_ed:
#             pos_clip_indices = random.sample(range(gt_st, gt_ed + 1), k=max_n)
#         else:
#             pos_clip_indices = [gt_st, gt_st]

#         neg_pool = list(range(0, gt_st)) + list(
#             range(gt_ed + 1, ctx_l)
#         )  # to fix bugs / works..?
#         try:
#             neg_clip_indices = random.sample(neg_pool, k=max_n)
#         except:
#             neg_clip_indices = pos_clip_indices

#         score_array = np.zeros(ctx_l)
#         score_array[gt_st : gt_ed + 1] = 1

#         return pos_clip_indices, neg_clip_indices, score_array

#### Dataset

1. Prepare Dataset
2. Prepare Dataloader

In [ ]:
class StartEndDataset(Dataset):
    """One line in data loaded from data_path."
    {
      "qid": 7803,
      "query": "Man in gray top walks from outside to inside.",
      "duration": 150,
      "vid": "RoripwjYFp8_360.0_510.0",
      "relevant_clip_ids": [13, 14, 15, 16, 17],
      "relevant_windows": [[26, 36]]
    }
    """

    def __init__(
        self,
        data_path: str,
        a_feat_dir: str,
        q_feat_dir: str,
        q_feat_type: str = "last_hidden_state",
        a_feat_type: str = "pann",
        max_q_l: int = 32,
        max_a_l: int = 75,
        ctx_mode: str = "video",
        clip_len: int = 2,
        max_windows: int = 5,
        span_loss_type: str = "l1",
        load_labels: bool = True,
    ) -> None:
        self.data_path = data_path
        self.a_feat_dir = a_feat_dir
        self.q_feat_dir = q_feat_dir
        self.q_feat_type = q_feat_type
        self.a_feat_type = a_feat_type

        if max_a_l == -1:
            max_a_l = 100000000

        if max_q_l == -1:
            max_q_l = 100

        self.max_q_l = max_q_l
        self.max_a_l = max_a_l

        self.ctx_mode = ctx_mode
        self.use_tef = "tef" in ctx_mode
        self.use_audio = "audio" in ctx_mode
        self.clip_len = clip_len
        self.max_windows = max_windows  # maximum number of windows to use as labels
        self.span_loss_type = span_loss_type
        self.load_labels = load_labels
        self.data = self.load_data()

    def load_data(self) -> List[Dict[str, Any]]:
        datalist = load_jsonl(self.data_path)
        return datalist

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        meta = self.data[index]

        model_inputs = dict()
        model_inputs["query_feat"] = self._get_query_feat_by_qid(
            meta["qid"]
        )  # (Dq, ) or (Lq, Dq)
        model_inputs["audio_feat"] = self._get_audio_feat_by_vid(meta["vid"])
        ctx_l = len(model_inputs["audio_feat"])

        if self.use_tef:
            duration = meta["duration"]  # Total video duration in seconds
            clip_indices = torch.arange(0, ctx_l, 1.0)  # [0, 1, 2, ..., ctx_l-1]
            tef_st = (clip_indices * self.clip_len) / duration  # Normalized start times
            tef_ed = (
                (clip_indices + 1) * self.clip_len
            ) / duration  # Normalized end times
            tef_ed = torch.clamp(tef_ed, max=1.0)  # Ensure it doesn't exceed 1.0
            tef = torch.stack([tef_st, tef_ed], dim=1)  # (ctx_l, 2)
            model_inputs["audio_feat"] = torch.cat(
                [model_inputs["audio_feat"], tef], dim=1
            )
        if self.load_labels:
            model_inputs["span_labels"] = self.get_span_labels(
                meta["relevant_windows"], ctx_l, meta["duration"]
            )
            (
                model_inputs["saliency_pos_labels"],
                model_inputs["saliency_neg_labels"],
                model_inputs["saliency_all_labels"],
            ) = self.get_saliency_labels_sub_as_query(
                meta["relevant_windows"][0], ctx_l
            )

        return dict(meta=meta, model_inputs=model_inputs)

    def get_span_labels(
        self, windows: List[List[float]], ctx_l: int, duration: float
    ) -> torch.Tensor:
        """
        windows: list([st, ed]) in seconds. E.g. [[26, 36]], corresponding st_ed clip_indices [[13, 17]] (inclusive)
            Note a maximum of `self.max_windows` windows are used.
        returns Tensor of shape (#windows, 2), each row is [center, width] normalized by video length
        """
        if len(windows) > self.max_windows:
            random.shuffle(windows)
            windows = windows[: self.max_windows]
        if self.span_loss_type == "l1":
            windows = torch.Tensor(windows) / duration  # normalized windows in xx
            windows = span_xx_to_cxw(windows)  # normalized windows in cxw
        elif self.span_loss_type == "ce":
            windows = torch.Tensor(
                [
                    [
                        int(w[0] / self.clip_len),
                        min(int(w[1] / self.clip_len), ctx_l) - 1,
                    ]
                    for w in windows
                ]
            ).long()  # inclusive
        else:
            raise NotImplementedError
        return windows

    def get_saliency_labels_sub_as_query(
        self, gt_window: List[float], ctx_l: int, max_n: int = 2
    ) -> Tuple[List[int], List[int], np.ndarray]:
        gt_st = int(gt_window[0] / self.clip_len)
        gt_ed = max(0, min(int(gt_window[1] / self.clip_len), ctx_l) - 1)

        if gt_st > gt_ed:
            gt_st = gt_ed

        if gt_st != gt_ed:
            pos_clip_indices = random.sample(range(gt_st, gt_ed + 1), k=max_n)
        else:
            pos_clip_indices = [gt_st, gt_st]

        neg_pool = list(range(0, gt_st)) + list(
            range(gt_ed + 1, ctx_l)
        )  # to fix bugs / works..?
        try:
            neg_clip_indices = random.sample(neg_pool, k=max_n)
        except:
            neg_clip_indices = pos_clip_indices

        score_array = np.zeros(ctx_l)
        score_array[gt_st : gt_ed + 1] = 1

        return pos_clip_indices, neg_clip_indices, score_array

    def get_saliency_labels(
        self,
        rel_clip_ids: List[int],
        scores: List[List[float]],
        ctx_l: int,
        max_n: int = 1,
        add_easy_negative: bool = True,
    ) -> Tuple[List[int], List[int]]:
        """Sum the scores from the three annotations, then take the two clips with the
        maximum scores as positive, and two with the minimum scores as negative.
        Args:
            rel_clip_ids: list(int), list of relevant clip ids
            scores: list([anno1_score, anno2_score, anno3_score]),
            ctx_l: int
            max_n: int, #clips to use as positive and negative, for easy and hard negative, respectively.
            add_easy_negative: bool, if True, sample eay negative outside the relevant_clip_ids.
        """
        # indices inside rel_clip_ids
        scores = np.array(scores)  # (#rel_clips, 3)
        agg_scores = np.sum(scores, 1)  # (#rel_clips, )
        sort_indices = np.argsort(agg_scores)  # increasing

        # indices in the whole video
        # the min(_, ctx_l-1) here is incorrect, but should not cause
        # much troubles since this should be rarely used.
        hard_pos_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[-max_n:]
        ]
        hard_neg_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[:max_n]
        ]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)) - set(rel_clip_ids))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices
        return pos_clip_indices, neg_clip_indices

    def _get_query_feat_by_qid(self, qid: int) -> np.ndarray:
        q_feat_path = join(self.q_feat_dir, f"qid{qid}.npz")
        q_feat = np.load(q_feat_path)["last_hidden_state"]
        return q_feat

    def _get_audio_feat_by_vid(self, vid: str) -> torch.Tensor:
        _feat_path = join(self.a_feat_dir, f"{vid}.npz")
        _feat = np.load(_feat_path)["features"][: self.max_a_l].astype(np.float32)
        _feat = l2_normalize_np_array(_feat)
        return torch.from_numpy(_feat)

In [ ]:
def start_end_collate(
    batch: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    batch_meta = [e["meta"] for e in batch]

    model_inputs_keys = batch[0]["model_inputs"].keys()
    batched_data = dict()
    for k in model_inputs_keys:
        if k == "span_labels":
            batched_data[k] = [
                dict(spans=e["model_inputs"]["span_labels"]) for e in batch
            ]
            continue
        if k in ["saliency_pos_labels", "saliency_neg_labels"]:
            batched_data[k] = torch.LongTensor([e["model_inputs"][k] for e in batch])
            continue
        if k == "saliency_all_labels":
            pad_data, mask_data = pad_sequences_1d(
                [e["model_inputs"][k] for e in batch],
                dtype=np.float32,
                fixed_length=None,
            )
            batched_data[k] = torch.tensor(pad_data, dtype=torch.float32)
            continue

        if batch[0]["model_inputs"][k].dtype == torch.float32:
            batched_data[k] = pad_sequences_1d(
                [e["model_inputs"][k] for e in batch],
                dtype=torch.float32,
                fixed_length=None,
            )
        else:
            batched_data[k] = pad_sequences_1d(
                [torch.from_numpy(e["model_inputs"][k]) for e in batch],
                dtype=torch.float32,
                fixed_length=None,
            )
    return batch_meta, batched_data


def prepare_batch_inputs(
    batched_model_inputs: Dict[str, Any],
    device: torch.device,
    non_blocking: bool = False,
) -> Tuple[Dict[str, torch.Tensor], Optional[Dict[str, Any]]]:
    model_inputs = dict(
        src_txt=batched_model_inputs["query_feat"][0].to(
            device, non_blocking=non_blocking
        ),
        src_txt_mask=batched_model_inputs["query_feat"][1].to(
            device, non_blocking=non_blocking
        ),
    )

    if "audio_feat" in batched_model_inputs:
        model_inputs["src_aud"] = batched_model_inputs["audio_feat"][0].to(
            device, non_blocking=non_blocking
        )
        model_inputs["src_aud_mask"] = batched_model_inputs["audio_feat"][1].to(
            device, non_blocking=non_blocking
        )

    targets = {}
    if "span_labels" in batched_model_inputs:
        targets["span_labels"] = [
            dict(spans=e["spans"].to(device, non_blocking=non_blocking))
            for e in batched_model_inputs["span_labels"]
        ]
    if "saliency_pos_labels" in batched_model_inputs:
        for name in ["saliency_pos_labels", "saliency_neg_labels"]:
            targets[name] = batched_model_inputs[name].to(
                device, non_blocking=non_blocking
            )

    if "saliency_all_labels" in batched_model_inputs:
        targets["saliency_all_labels"] = batched_model_inputs["saliency_all_labels"].to(
            device, non_blocking=non_blocking
        )

    targets = None if len(targets) == 0 else targets
    return model_inputs, targets

In [ ]:
config_path = LOCAL_DIR / "config" / "config_pretraining.yml"
opt = read_yaml(config_path)
opt

In [ ]:
FEATURES_DIR

In [ ]:
opt["train_path"] = str(PREPROCESSED_DIR / "clotho_moment_valid_train.jsonl")
opt["val_path"] = str(PREPROCESSED_DIR / "clotho_moment_valid_val.jsonl")
opt["test_path"] = str(PREPROCESSED_DIR / "clotho_moment_valid_test.jsonl")


# opt['a_feat_dir'] = str(FEATURES_DIR / "clotho-moment" / "clap")
# opt['t_feat_dir'] = str(FEATURES_DIR / "clotho-moment" / "clap_text")


a_feat_dir = str(DATA_DIR / "clotho-moment" / "clap")
q_feat_dir = str(DATA_DIR / "clotho-moment" / "clap_text")
opt["a_feat_dir"] = a_feat_dir
opt["t_feat_dir"] = q_feat_dir

In [ ]:
q_np_feat = np.load(os.path.join(q_feat_dir, os.listdir(q_feat_dir)[0]))


q_np_feat

In [ ]:
q_np_feat["last_hidden_state"].shape

In [ ]:
a_np_feat = np.load(os.path.join(a_feat_dir, os.listdir(a_feat_dir)[3]))


a_np_feat.keys()

In [ ]:
a_np_feat["features"].shape

In [ ]:
a_np_feat["proj_features"].shape

In [ ]:
dataset_config = EasyDict(
    data_path=opt.get("train_path"),
    ctx_mode=opt.get("ctx_mode"),
    a_feat_dir=opt.get("a_feat_dir"),
    q_feat_dir=opt.get("t_feat_dir"),
    q_feat_type="last_hidden_state",
    a_feat_type=opt.get("a_feat_type"),
    max_q_l=opt.get("max_q_l"),
    max_a_l=opt.get("max_a_l"),
    clip_len=opt.get("clip_length"),
    max_windows=opt.get("max_windows"),
    span_loss_type=opt.get("span_loss_type"),
    load_labels=True,
)

In [ ]:
train_dataset = StartEndDataset(
    **dataset_config,
)

In [ ]:
dataset_config

In [ ]:
len(os.listdir(a_feat_dir))

In [ ]:
len(os.listdir(q_feat_dir))

In [ ]:
len(train_dataset)

In [ ]:
train_sample = train_dataset[512]

In [ ]:
train_sample

### Core Model


In [ ]:
class MLP(nn.Module):
    """Very simple multi-layer perceptron (also called FFN)"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        """Forward pass through the MLP.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor.
        """
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


class LinearLayer(nn.Module):
    """linear layer configurable with layer normalization, dropout, ReLU."""

    def __init__(self, in_hsz, out_hsz, layer_norm=True, dropout=0.1, relu=True):
        super(LinearLayer, self).__init__()
        self.relu = relu
        self.layer_norm = layer_norm
        if layer_norm:
            self.LayerNorm = nn.LayerNorm(in_hsz)
        layers = [nn.Dropout(dropout), nn.Linear(in_hsz, out_hsz)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """(N, L, D)"""
        if self.layer_norm:
            x = self.LayerNorm(x)
        x = self.net(x)
        if self.relu:
            x = F.relu(x, inplace=True)
        return x  # (N, L, D)

#### Positional Encoding

In [ ]:
class TrainablePositionalEncoding(nn.Module):
    """Construct the embeddings from word, position and token_type embeddings."""

    def __init__(self, max_position_embeddings, hidden_size, dropout=0.1):
        super(TrainablePositionalEncoding, self).__init__()
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_feat):
        """
        Args:
            input_feat: (N, L, D)
        """
        bsz, seq_length = input_feat.shape[:2]
        position_ids = torch.arange(
            seq_length, dtype=torch.long, device=input_feat.device
        )
        position_ids = position_ids.unsqueeze(0).repeat(bsz, 1)  # (N, L)

        position_embeddings = self.position_embeddings(position_ids)

        embeddings = self.LayerNorm(input_feat + position_embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images. (To 1D sequences)
    """

    def __init__(
        self, num_pos_feats=64, temperature=10000, normalize=False, scale=None
    ):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x, mask):
        """
        Args:
            x: torch.tensor, (batch_size, L, d)
            mask: torch.tensor, (batch_size, L), with 1 as valid

        Returns:

        """
        assert mask is not None
        x_embed = mask.cumsum(1, dtype=torch.float32)  # (bsz, L)
        if self.normalize:
            eps = 1e-6
            x_embed = x_embed / (x_embed[:, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, None] / dim_t  # (bsz, L, num_pos_feats)
        pos_x = torch.stack(
            (pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3
        ).flatten(2)  # (bsz, L, num_pos_feats*2)
        return pos_x  # .permute(0, 2, 1)  # (bsz, num_pos_feats*2, L)


class PositionEmbeddingLearned(nn.Module):
    """
    Absolute pos embedding, learned.
    """

    def __init__(self, num_pos_feats=256):
        super().__init__()
        self.row_embed = nn.Embedding(50, num_pos_feats)
        self.col_embed = nn.Embedding(50, num_pos_feats)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)

    def forward(self, x, mask):
        h, w = x.shape[-2:]
        i = torch.arange(w, device=x.device)
        j = torch.arange(h, device=x.device)
        x_emb = self.col_embed(i)
        y_emb = self.row_embed(j)
        pos = (
            torch.cat(
                [
                    x_emb.unsqueeze(0).repeat(h, 1, 1),
                    y_emb.unsqueeze(1).repeat(1, w, 1),
                ],
                dim=-1,
            )
            .permute(2, 0, 1)
            .unsqueeze(0)
            .repeat(x.shape[0], 1, 1, 1)
        )
        return pos


def build_position_encoding(args):
    N_steps = args.hidden_dim
    if args.position_embedding in ("v2", "sine"):
        # TODO find a better way of exposing other arguments
        position_embedding = PositionEmbeddingSine(N_steps, normalize=True)
    # elif args.position_embedding in ('v3', 'learned'):
    #     position_embedding = PositionEmbeddingLearned(N_steps)
    else:
        raise ValueError(f"not supported {args.position_embedding}")

    txt_pos_embed = TrainablePositionalEncoding(
        max_position_embeddings=args.max_q_l,
        hidden_size=args.hidden_dim,
        dropout=args.input_dropout,
    )
    return position_embedding, txt_pos_embed

#### Encoder

#### Decoder

#### Criteria

In [ ]:
class SetCriterion(nn.Module):
    """This class computes the loss for DETR.
    The process happens in two steps:
        1) we compute hungarian assignment between ground truth boxes and the outputs of the model
        2) we supervise each pair of matched ground-truth / prediction (supervise class and box)
    """

    def __init__(
        self,
        matcher,
        weight_dict,
        eos_coef,
        losses,
        span_loss_type,
        max_a_l,
        saliency_margin=1,
    ):
        """Create the criterion.
        Parameters:
            matcher: module able to compute a matching between targets and proposals
            weight_dict: dict containing as key the names of the losses and as values their relative weight.
            eos_coef: relative classification weight applied to the no-object category
            losses: list of all the losses to be applied. See get_loss for list of available losses.
            span_loss_type: str, [l1, ce]
            max_v_l: int,
            saliency_margin: float
        """
        super().__init__()
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.losses = losses
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        self.saliency_margin = saliency_margin

        # foreground and background classification
        self.foreground_label = 0
        self.background_label = 1
        self.eos_coef = eos_coef
        empty_weight = torch.ones(2)
        empty_weight[-1] = (
            self.eos_coef
        )  # lower weight for background (index 1, foreground index 0)
        self.register_buffer("empty_weight", empty_weight)

    def loss_spans(self, outputs, targets, indices):
        """Compute the losses related to the bounding boxes, the L1 regression loss and the GIoU loss
        targets dicts must contain the key "spans" containing a tensor of dim [nb_tgt_spans, 2]
        The target spans are expected in format (center_x, w), normalized by the image size.
        """
        assert "pred_spans" in outputs
        targets = targets["span_labels"]
        idx = self._get_src_permutation_idx(indices)
        src_spans = outputs["pred_spans"][idx]  # (#spans, max_v_l * 2)
        tgt_spans = torch.cat(
            [t["spans"][i] for t, (_, i) in zip(targets, indices)], dim=0
        )  # (#spans, 2)
        if self.span_loss_type == "l1":
            loss_span = F.l1_loss(src_spans, tgt_spans, reduction="none")
            loss_giou = 1 - torch.diag(
                generalized_temporal_iou(
                    span_cxw_to_xx(src_spans), span_cxw_to_xx(tgt_spans)
                )
            )
        else:  # ce
            n_spans = src_spans.shape[0]
            src_spans = src_spans.view(n_spans, 2, self.max_v_l).transpose(1, 2)
            loss_span = F.cross_entropy(src_spans, tgt_spans, reduction="none")
            loss_giou = loss_span.new_zeros([1])

        losses = {}
        losses["loss_span"] = loss_span.mean()
        losses["loss_giou"] = loss_giou.mean()
        return losses

    def loss_labels(self, outputs, targets, indices, log=True):
        """Classification loss (NLL)
        targets dicts must contain the key "labels" containing a tensor of dim [nb_target_boxes]
        """
        # TODO add foreground and background classifier.  use all non-matched as background.
        assert "pred_logits" in outputs
        src_logits = outputs["pred_logits"]  # (batch_size, #queries, #classes=2)
        # idx is a tuple of two 1D tensors (batch_idx, src_idx), of the same length == #objects in batch
        idx = self._get_src_permutation_idx(indices)
        target_classes = torch.full(
            src_logits.shape[:2],
            self.background_label,
            dtype=torch.int64,
            device=src_logits.device,
        )  # (batch_size, #queries)
        target_classes[idx] = self.foreground_label

        loss_ce = F.cross_entropy(
            src_logits.transpose(1, 2),
            target_classes,
            self.empty_weight,
            reduction="none",
        )
        losses = {"loss_label": loss_ce.mean()}

        if log:
            # TODO this should probably be a separate loss, not hacked in this one here
            losses["class_error"] = (
                100 - accuracy(src_logits[idx], self.foreground_label)[0]
            )
        return losses

    def loss_saliency(self, outputs, targets, indices, log=True):
        """higher scores for positive clips"""
        if "saliency_pos_labels" not in targets:
            return {"loss_saliency": 0}

        aud_token_mask = outputs["audio_mask"]

        # Neg pair loss
        saliency_scores_neg = outputs["saliency_scores_neg"].clone()  # (N, L)

        loss_neg_pair = (
            (-torch.log(1.0 - torch.sigmoid(saliency_scores_neg)) * aud_token_mask)
            .sum(dim=1)
            .mean()
        )

        saliency_scores = outputs["saliency_scores"].clone()  # (N, L)
        saliency_contrast_label = targets["saliency_all_labels"]

        saliency_scores = torch.cat([saliency_scores, saliency_scores_neg], dim=1)
        saliency_contrast_label = torch.cat(
            [saliency_contrast_label, torch.zeros_like(saliency_contrast_label)], dim=1
        )

        aud_token_mask = aud_token_mask.repeat([1, 2])
        saliency_scores = (
            aud_token_mask * saliency_scores + (1.0 - aud_token_mask) * -1e3
        )

        tau = 0.5
        loss_rank_contrastive = 0.0

        # for rand_idx in range(1, 13, 3):
        #     # 1, 4, 7, 10 --> 5 stages
        for rand_idx in range(1, 12):
            drop_mask = ~(saliency_contrast_label > 100)  # no drop
            pos_mask = (
                saliency_contrast_label >= rand_idx
            )  # positive when equal or higher than rand_idx

            if torch.sum(pos_mask) == 0:  # no positive sample
                continue
            else:
                batch_drop_mask = (
                    torch.sum(pos_mask, dim=1) > 0
                )  # negative sample indicator

            # drop higher ranks
            cur_saliency_scores = saliency_scores * drop_mask / tau + ~drop_mask * -1e3

            # numerical stability
            logits = (
                cur_saliency_scores
                - torch.max(cur_saliency_scores, dim=1, keepdim=True)[0]
            )

            # softmax
            exp_logits = torch.exp(logits)
            log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

            mean_log_prob_pos = (pos_mask * log_prob * aud_token_mask).sum(1) / (
                pos_mask.sum(1) + 1e-6
            )

            loss = -mean_log_prob_pos * batch_drop_mask

            loss_rank_contrastive = loss_rank_contrastive + loss.mean()

        loss_rank_contrastive = loss_rank_contrastive / 12

        saliency_scores = outputs["saliency_scores"]  # (N, L)
        pos_indices = targets["saliency_pos_labels"]  # (N, #pairs)
        neg_indices = targets["saliency_neg_labels"]  # (N, #pairs)
        num_pairs = pos_indices.shape[1]  # typically 2 or 4
        batch_indices = torch.arange(len(saliency_scores)).to(saliency_scores.device)
        pos_scores = torch.stack(
            [
                saliency_scores[batch_indices, pos_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        neg_scores = torch.stack(
            [
                saliency_scores[batch_indices, neg_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        loss_saliency = (
            torch.clamp(self.saliency_margin + neg_scores - pos_scores, min=0).sum()
            / (len(pos_scores) * num_pairs)
            * 2
        )  # * 2 to keep the loss the same scale

        loss_saliency = loss_saliency + loss_rank_contrastive + loss_neg_pair
        return {"loss_saliency": loss_saliency}

    def _get_src_permutation_idx(self, indices):
        """Permutes predictions following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, src_idx
        """
        # permute predictions following indices
        batch_idx = torch.cat(
            [torch.full_like(src, i) for i, (src, _) in enumerate(indices)]
        )
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx  # two 1D tensors of the same length

    def _get_tgt_permutation_idx(self, indices):
        """Permutes targets following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, tgt_idx
        """
        # permute targets following indices
        batch_idx = torch.cat(
            [torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)]
        )
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx

    def get_loss(self, loss, outputs, targets, indices, **kwargs):
        """Retrieves the loss function for the given loss type.

        Args:
            loss (str): Type of loss.
            outputs: Model outputs.
            targets: Ground truth targets.
            indices: Matched indices.
            **kwargs: Additional arguments.

        Returns:
            dict: Loss values.
        """
        loss_map = {
            "spans": self.loss_spans,
            "labels": self.loss_labels,
            "saliency": self.loss_saliency,
        }
        assert loss in loss_map, f"do you really want to compute {loss} loss?"
        return loss_map[loss](outputs, targets, indices, **kwargs)

    def forward(self, outputs, targets):
        """This performs the loss computation.
        Parameters:
             outputs: dict of tensors, see the output specification of the model for the format
             targets: list of dicts, such that len(targets) == batch_size.
                      The expected keys in each dict depends on the losses applied, see each loss' doc
        """
        outputs_without_aux = {k: v for k, v in outputs.items() if k != "aux_outputs"}

        # Retrieve the matching between the outputs of the last layer and the targets
        # list(tuples), each tuple is (pred_span_indices, tgt_span_indices)

        indices = self.matcher(outputs_without_aux, targets)
        losses_target = self.losses

        # Compute all the requested losses
        losses = {}
        for loss in losses_target:
            losses.update(self.get_loss(loss, outputs, targets, indices))

        # In case of auxiliary losses, we repeat this process with the output of each intermediate layer.
        if "aux_outputs" in outputs:
            for i, aux_outputs in enumerate(outputs["aux_outputs"]):
                indices = self.matcher(aux_outputs, targets)
                losses_target = self.losses

                for loss in losses_target:
                    if "saliency" == loss:  # skip as it is only in the top layer
                        continue
                    kwargs = {}
                    l_dict = self.get_loss(
                        loss, aux_outputs, targets, indices, **kwargs
                    )
                    l_dict = {k + f"_{i}": v for k, v in l_dict.items()}
                    losses.update(l_dict)
        return losses

### Training


### Evaluating